# Rice Disease Model Comparison

Final held-out evaluation for YOLO11n-cls and MobileNetV3-Large. This notebook uses only the approved manifests and class_mapping.json. It refuses to evaluate until both best checkpoints and training environment records exist.

Test data is used here only after both training notebooks have completed. No training or hyperparameter tuning occurs in this notebook. Set RUN_TEST_EVALUATION=True explicitly to read test.csv and run the final evaluation.

Both models receive the same test image paths in the same order and the same class IDs. Preprocessing is architecture-specific and recorded separately: YOLO uses Ultralytics classification preprocessing; MobileNet uses its ImageNet weight preset. Latency includes decode/preprocessing and batch-one inference. Parameter count and checkpoint file size are reported separately.

In [ ]:
%pip -q install ultralytics==8.4.215 pandas scikit-learn seaborn matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, platform, random, time
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torchvision import models
from torchvision.models import MobileNet_V3_Large_Weights
from ultralytics import YOLO
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

SEED = 42
IMG_SIZE = 224
BATCH_SIZE_FOR_LATENCY = 1
RUN_TEST_EVALUATION = False  # Set True only after both model notebooks have saved best checkpoints.
DRIVE_ROOT = Path('/content/drive/MyDrive/Krishok_AI')
DATA_ROOT = DRIVE_ROOT / 'data/processed/rice_classification'
MANIFEST_DIR = DATA_ROOT / 'manifests'
RUNS_ROOT = DRIVE_ROOT / 'runs/rice_classification'
YOLO_RUN_DIR = RUNS_ROOT / 'yolo11n_cls'
MOBILE_RUN_DIR = DRIVE_ROOT / 'AI/experiments/MobileNetV3_Rice'
COMPARISON_DIR = RUNS_ROOT / 'comparison'
YOLO_BEST = YOLO_RUN_DIR / 'weights' / 'best.pt'
MOBILE_BEST = MOBILE_RUN_DIR / 'best_model.pt'
CLASS_MAP_PATH = MANIFEST_DIR / 'class_mapping.json'
TRAIN_CSV = MANIFEST_DIR / 'train.csv'
VAL_CSV = MANIFEST_DIR / 'val.csv'
TEST_CSV = MANIFEST_DIR / 'test.csv'
CROSS_REVIEW_CSV = MANIFEST_DIR / 'cross_class_review.csv'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
assert torch.cuda.is_available(), 'Select a Colab GPU runtime for the requested benchmark.'
device = torch.device('cuda:0')
print({'torch': torch.__version__, 'ultralytics': __import__('ultralytics').__version__,
       'gpu': torch.cuda.get_device_name(0), 'seed': SEED})

## Held-out evaluation gate

The code below does not open test.csv unless RUN_TEST_EVALUATION is True. It first verifies both checkpoints and that their recorded train/validation manifest digests match the current approved manifests. Training notebooks must finish before enabling this cell.

In [ ]:
if not RUN_TEST_EVALUATION:
    print('Test evaluation not started. Confirm both best checkpoints exist, then set RUN_TEST_EVALUATION=True.')
else:
    required = [YOLO_BEST, MOBILE_BEST,
                YOLO_RUN_DIR / 'environment.json', MOBILE_RUN_DIR / 'environment.json']
    missing = [str(p) for p in required if not p.is_file()]
    assert not missing, f'Both training runs must finish first. Missing: {missing}'

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            for block in iter(lambda: f.read(1024 * 1024), b''):
                h.update(block)
        return h.hexdigest()

    class_map = json.loads(CLASS_MAP_PATH.read_text(encoding='utf-8'))
    classes = class_map['classes']
    class_to_id = {item['class_label']: int(item['class_id']) for item in classes}
    class_names = [label for label, idx in sorted(class_to_id.items(), key=lambda item: item[1])]
    assert class_map['uses_yolo_annotation_class_ids'] is False
    yolo_env = json.loads((YOLO_RUN_DIR / 'environment.json').read_text(encoding='utf-8'))
    mobile_env = json.loads((MOBILE_RUN_DIR / 'environment.json').read_text(encoding='utf-8'))
    train_digest, val_digest = sha256_file(TRAIN_CSV), sha256_file(VAL_CSV)
    assert yolo_env['train_manifest_sha256'] == mobile_env['train_manifest_sha256'] == train_digest
    assert yolo_env['val_manifest_sha256'] == mobile_env['val_manifest_sha256'] == val_digest

    train_df = pd.read_csv(TRAIN_CSV)
    val_df = pd.read_csv(VAL_CSV)
    test_df = pd.read_csv(TEST_CSV)
    assert len(train_df) == 6827 and len(val_df) == 1463 and len(test_df) == 1463
    assert set(train_df.class_label) == set(val_df.class_label) == set(test_df.class_label) == set(class_to_id)
    for split, frame in [('train', train_df), ('val', val_df), ('test', test_df)]:
        assert frame.image_path.is_unique
        expected_ids = frame.class_label.map(class_to_id).astype(int)
        assert frame.class_id.astype(int).equals(expected_ids), f'Class mapping mismatch in {split}'
        missing_paths = [p for p in frame.image_path if not (DATA_ROOT / p).is_file()]
        assert not missing_paths, f'Missing {split} image paths: {missing_paths[:3]}'
    split_groups = [set(frame.duplicate_group_id) for frame in (train_df, val_df, test_df)]
    assert not (split_groups[0] & split_groups[1])
    assert not (split_groups[0] & split_groups[2])
    assert not (split_groups[1] & split_groups[2])
    review = pd.read_csv(CROSS_REVIEW_CSV)
    unresolved = set(review.loc[review.review_status == 'unresolved_excluded', 'group_id'])
    assert not (unresolved & set(test_df.duplicate_group_id))
    assert not (unresolved & set(train_df.duplicate_group_id))
    assert not (unresolved & set(val_df.duplicate_group_id))
    test_paths = [DATA_ROOT / p for p in test_df.image_path]
    y_true = test_df.class_id.astype(int).to_numpy()
    assert len(test_paths) == len(y_true) == 1463
    print({'test_images': len(test_paths), 'classes': class_to_id,
           'test_manifest_sha256': sha256_file(TEST_CSV),
           'unresolved_groups_excluded': len(unresolved)})

## Load best checkpoints and confirm identical class order

In [ ]:
if RUN_TEST_EVALUATION:
    yolo = YOLO(str(YOLO_BEST))
    yolo_names = [yolo.names[i] for i in range(len(yolo.names))]
    assert yolo_names == class_names, f'YOLO class order differs: {yolo_names} vs {class_names}'

    mobile_weights = MobileNet_V3_Large_Weights.DEFAULT
    eval_transform = mobile_weights.transforms()
    mobile = models.mobilenet_v3_large(weights=None)
    mobile.classifier[-1] = nn.Linear(mobile.classifier[-1].in_features, len(class_names))
    mobile_checkpoint = torch.load(MOBILE_BEST, map_location=device, weights_only=True)
    assert mobile_checkpoint['class_to_id'] == class_to_id
    mobile.load_state_dict(mobile_checkpoint['model_state_dict'])
    mobile.to(device).eval()
    print('YOLO and MobileNet class mappings match the approved mapping.')

## Final test evaluation and batch-one latency

The same test paths and class IDs are supplied to each model. Warm-up uses validation images only. The timer includes image decode, each model's documented preprocessing, and batch-one prediction; preprocessing differs by architecture.

In [ ]:
if RUN_TEST_EVALUATION:
    val_paths = [DATA_ROOT / p for p in val_df.image_path]
    for path in val_paths[:8]:
        _ = yolo.predict(str(path), imgsz=IMG_SIZE, batch=1, device=0, verbose=False)
        with Image.open(path) as image:
            tensor = eval_transform(image.convert('RGB')).unsqueeze(0).to(device)
        with torch.inference_mode():
            _ = mobile(tensor)
    torch.cuda.synchronize()

    yolo_pred, yolo_latency_ms, yolo_forward_ms = [], [], []
    for path in test_paths:
        torch.cuda.synchronize()
        started = time.perf_counter()
        result = yolo.predict(str(path), imgsz=IMG_SIZE, batch=1, device=0, verbose=False)[0]
        torch.cuda.synchronize()
        yolo_latency_ms.append((time.perf_counter() - started) * 1000)
        yolo_pred.append(int(result.probs.top1))
        speed = getattr(result, 'speed', {}) or {}
        speed_value = speed.get('inference', np.nan)
        yolo_forward_ms.append(float(speed_value) if speed_value is not None else np.nan)
    assert len(yolo_pred) == len(test_paths)

    mobile_pred, mobile_latency_ms, mobile_forward_ms = [], [], []
    for path in test_paths:
        torch.cuda.synchronize()
        started = time.perf_counter()
        with Image.open(path) as image:
            tensor = eval_transform(image.convert('RGB')).unsqueeze(0).to(device)
        torch.cuda.synchronize()
        after_preprocess = time.perf_counter()
        with torch.inference_mode():
            logits = mobile(tensor)
        torch.cuda.synchronize()
        finished = time.perf_counter()
        mobile_pred.append(int(logits.argmax(1).item()))
        mobile_latency_ms.append((finished - started) * 1000)
        mobile_forward_ms.append((finished - after_preprocess) * 1000)
    assert len(mobile_pred) == len(test_paths)
    print('Finished test predictions for identical paths:', len(test_paths))

## Metrics, confusion matrices, and comparison CSV

In [ ]:
if RUN_TEST_EVALUATION:
    from sklearn.metrics import classification_report
    import matplotlib.pyplot as plt
    import seaborn as sns

    COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    model_outputs = {
        'YOLO11n-cls': (np.asarray(yolo_pred), np.asarray(yolo_latency_ms), np.asarray(yolo_forward_ms),
                        YOLO_BEST, sum(p.numel() for p in yolo.model.parameters())),
        'MobileNetV3-Large': (np.asarray(mobile_pred), np.asarray(mobile_latency_ms), np.asarray(mobile_forward_ms),
                              MOBILE_BEST, sum(p.numel() for p in mobile.parameters())),
    }
    comparison_rows, class_rows, prediction_rows = [], [], []
    for model_name, (pred, latency, forward_latency, checkpoint_path, param_count) in model_outputs.items():
        accuracy = accuracy_score(y_true, pred)
        macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
            y_true, pred, labels=list(range(len(class_names))), average='macro', zero_division=0)
        per_class = precision_recall_fscore_support(
            y_true, pred, labels=list(range(len(class_names))), average=None, zero_division=0)
        cm = confusion_matrix(y_true, pred, labels=list(range(len(class_names))))
        for idx, label in enumerate(class_names):
            class_rows.append({'model': model_name, 'class_id': idx, 'class_name': label,
                               'precision': per_class[0][idx], 'recall': per_class[1][idx],
                               'f1': per_class[2][idx], 'support': int(per_class[3][idx])})
        for row, pred_id in zip(test_df.itertuples(index=False), pred):
            prediction_rows.append({'model': model_name, 'image_path': row.image_path,
                                    'true_class_id': int(row.class_id), 'true_class': row.class_label,
                                    'predicted_class_id': int(pred_id), 'predicted_class': class_names[int(pred_id)]})
        safe_name = model_name.lower().replace('-', '_').replace(' ', '_')
        plt.figure(figsize=(8, 7))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
        plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'{model_name} confusion matrix')
        plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0); plt.tight_layout()
        plt.savefig(COMPARISON_DIR / f'{safe_name}_confusion_matrix.png', dpi=160)
        plt.show()
        comparison_rows.append({
            'model': model_name, 'accuracy': accuracy, 'macro_precision': macro_p,
            'macro_recall': macro_r, 'macro_f1': macro_f1,
            'parameters': int(param_count), 'checkpoint_bytes': checkpoint_path.stat().st_size,
            'checkpoint_MiB': checkpoint_path.stat().st_size / (1024 ** 2),
            'end_to_end_latency_mean_ms': float(np.mean(latency)),
            'end_to_end_latency_median_ms': float(np.median(latency)),
            'end_to_end_latency_p95_ms': float(np.percentile(latency, 95)),
            'model_forward_mean_ms': float(np.nanmean(forward_latency)),
            'test_images': len(y_true), 'test_manifest_sha256': sha256_file(TEST_CSV),
        })
    pd.DataFrame(comparison_rows).to_csv(COMPARISON_DIR / 'model_comparison.csv', index=False)
    pd.DataFrame(class_rows).to_csv(COMPARISON_DIR / 'per_class_metrics.csv', index=False)
    pd.DataFrame(prediction_rows).to_csv(COMPARISON_DIR / 'test_predictions.csv', index=False)
    display(pd.DataFrame(comparison_rows))
    display(pd.DataFrame(class_rows))
    print('Metrics written to:', COMPARISON_DIR)

## Loss-curve comparison and run record

In [ ]:
if RUN_TEST_EVALUATION:
    import matplotlib.pyplot as plt
    yolo_results_path = YOLO_RUN_DIR / 'results.csv'
    mobile_history_path = MOBILE_RUN_DIR / 'training_history.csv'
    assert yolo_results_path.is_file() and mobile_history_path.is_file()
    yhist = pd.read_csv(yolo_results_path)
    yhist.columns = [c.strip() for c in yhist.columns]
    mhist = pd.read_csv(mobile_history_path)
    y_train = next(c for c in yhist.columns if c.lower() == 'train/loss')
    y_val = next(c for c in yhist.columns if c.lower() == 'val/loss')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(yhist[y_train], label='Train'); axes[0].plot(yhist[y_val], label='Validation')
    axes[0].set_title('YOLO11n-cls'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(mhist.train_loss, label='Train'); axes[1].plot(mhist.val_loss, label='Validation')
    axes[1].set_title('MobileNetV3-Large'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend(); axes[1].grid(True)
    fig.tight_layout()
    fig.savefig(COMPARISON_DIR / 'training_validation_loss_curves.png', dpi=160)
    plt.show()
    run_record = {
        'status': 'evaluated', 'seed': SEED, 'test_images': len(test_df),
        'test_manifest_sha256': sha256_file(TEST_CSV), 'class_mapping': class_to_id,
        'preprocessing': {
            'YOLO11n-cls': 'Ultralytics classification transforms at 224 px; built-in resize/crop/tensor and model defaults.',
            'MobileNetV3-Large': 'Torchvision ImageNet weight preset for evaluation; resize/center-crop 224 and ImageNet mean/std.',
        },
        'torch': torch.__version__, 'ultralytics': __import__('ultralytics').__version__,
        'gpu': torch.cuda.get_device_name(0),
        'test_used_for_training_or_tuning': False,
    }
    (COMPARISON_DIR / 'comparison_run.json').write_text(json.dumps(run_record, indent=2), encoding='utf-8')
    print('Loss curves and run record saved to:', COMPARISON_DIR)
else:
    print('No test metrics or curves generated. Enable RUN_TEST_EVALUATION after both training runs finish.')

## Saved artifacts

The comparison directory contains model_comparison.csv, per_class_metrics.csv, one confusion matrix per model, test_predictions.csv, training_validation_loss_curves.png, and comparison_run.json. The CSVs contain measured values from this run only; this notebook has no prefilled accuracy claims.